1.도시재생/정비사업 사업방식 추천기

In [1]:
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

load_dotenv()

# 1. 모델 설정
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# 2. PromptTemplate 작성
template = """
당신은 도시재생 및 정비사업 분야의 전문 컨설턴트입니다.
입력된 대상지의 지구 특성을 바탕으로 가장 적합한 정비사업 유형을 추천하고, 해당 사업의 핵심 지정 요건 및 추진 가이드를 작성해주세요.

지구 특성: {site_info}

[출력 형식]
추천 정비사업: <사업 방식명 (예: 재개발사업, 재건축사업, 가로주택정비사업, 소규모재개발 등)>
추천 이유: <선정이유 요약>
핵심 필요 요건:
1. <요건 1>
2. <요건 2>
사업 추진 가이드:
1. <단계 1>
2. <단계 2>
3. <단계 3>
"""

prompt = PromptTemplate.from_template(template)

# 3. LCEL 파이프라인 결합
chain = prompt | llm | StrOutputParser()

# 4. 실행 및 결과 출력
site_input = "노후도 70%, 20년 이상 건축물 비율 80%, 도로율 10% 미만, 구역 면적 10,000㎡ 이하"
result = chain.invoke({"site_info": site_input})

print(f"입력 지구 특성: \"{site_input}\"")
print("\n--- 도시정비사업 컨설팅 결과 ---")
print(result)

입력 지구 특성: "노후도 70%, 20년 이상 건축물 비율 80%, 도로율 10% 미만, 구역 면적 10,000㎡ 이하"

--- 도시정비사업 컨설팅 결과 ---
추천 정비사업: 소규모재개발

추천 이유: 대상지는 노후도가 70%로 상당히 높은 편이며, 20년 이상 된 건축물 비율이 80%에 달합니다. 또한 도로율이 10% 미만으로 낮아 인프라가 부족한 상황입니다. 이러한 특성을 고려할 때, 소규모재개발은 상대적으로 적은 면적(10,000㎡ 이하)에서 주민의 동의를 얻어 추진할 수 있는 유연한 사업 방식으로 적합합니다.

핵심 필요 요건:
1. 사업구역 내 주민 3분의 2 이상의 동의 확보
2. 사업구역 내 건축물의 노후도 및 안전성 평가

사업 추진 가이드:
1. 주민 설명회 개최: 사업의 필요성과 이점을 주민에게 설명하고, 의견을 수렴하여 주민의 참여를 유도합니다.
2. 주민 동의서 수집: 사업 추진을 위한 주민 동의를 확보하기 위해 동의서를 수집합니다.
3. 사업 계획 수립 및 인허가 절차 진행: 주민 동의가 확보되면, 구체적인 사업 계획을 수립하고 관련 인허가 절차를 진행하여 사업을 본격적으로 추진합니다.


2.신도시 단계별 교통 대책 수립 시스템

In [2]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

# --------------------------------------------------
# 1단계 체인: 교통 진단 및 문제점 도출
# --------------------------------------------------
prompt_step1 = ChatPromptTemplate.from_messages([
    ("system", "당신은 신도시 교통계획 전문가입니다. 제공된 인구 및 통근 데이터를 바탕으로 해당 신도시의 핵심 교통 병목 구간과 주요 문제를 3가지로 진단하세요."),
    ("user", "{traffic_data}")
])

chain1 = prompt_step1 | llm | StrOutputParser()

# --------------------------------------------------
# 2단계 체인: 단기 및 장기 교통대책 수립
# --------------------------------------------------
prompt_step2 = ChatPromptTemplate.from_messages([
    ("system", "당신은 국가교통정책 수립 전문가입니다. 전달받은 교통 진단 내용을 바탕으로 단기 대책(S-BRT 신설, ITS 차선 제어, 버스 노선 확충 등)과 장기 대책(IC 연결, 도로 신설, 철도망 연장 등)을 구체적으로 수립해주세요."),
    ("user", """
진단된 교통 문제:
{diagnosis}

다음 양식에 따라 단기/장기 대책을 작성하세요:
[단기 교통 대책 (1~2년 내 시행)]
1. ...
2. ...

[장기 교통 대책 (5~10년 목표)]
1. ...
2. ...
""")
])

chain2 = prompt_step2 | llm | StrOutputParser()

# --------------------------------------------------
# 실행 및 결과 연결
# --------------------------------------------------
traffic_input = "계획인구 10만 명, 서울 방향 출퇴근 비율 70%, 주요 간선도로 출퇴근 시간대 평균 속도 15km/h(심각한 정체), 대중교통 분담률 20% 미만"

# 1단계 실행
diagnosis_result = chain1.invoke({"traffic_data": traffic_input})
print(f"입력 데이터: \"{traffic_input}\"\n")
print("=== 1단계: 교통 진단 결과 ===")
print(diagnosis_result)
print("\n" + "="*40 + "\n")

# 2단계 실행
policy_result = chain2.invoke({"diagnosis": diagnosis_result})
print("=== 2단계: 단계별 교통 대책 수립 ===")
print(policy_result)

입력 데이터: "계획인구 10만 명, 서울 방향 출퇴근 비율 70%, 주요 간선도로 출퇴근 시간대 평균 속도 15km/h(심각한 정체), 대중교통 분담률 20% 미만"

=== 1단계: 교통 진단 결과 ===
제공된 인구 및 통근 데이터를 바탕으로 신도시의 핵심 교통 병목 구간과 주요 문제를 다음과 같이 진단할 수 있습니다.

1. **고속도로 및 간선도로의 심각한 정체**:
   - 출퇴근 시간대 평균 속도가 15km/h로, 이는 교통 흐름이 매우 나쁜 상태를 나타냅니다. 계획인구 10만 명 중 70%인 7만 명이 서울 방향으로 출퇴근을 하게 되면, 도로에 집중되는 차량 수가 급증하여 심각한 정체가 발생할 것입니다. 이는 운전자의 스트레스 증가와 교통사고 위험을 높이는 요인이 됩니다.

2. **대중교통 이용률 저조**:
   - 대중교통 분담률이 20% 미만이라는 것은 대중교통 시스템이 신도시 주민들에게 충분히 매력적이지 않거나, 접근성이 떨어진다는 것을 의미합니다. 이는 개인 차량 의존도를 높이고, 결과적으로 도로 혼잡도를 더욱 악화시킬 수 있습니다. 대중교통의 서비스 개선이 시급합니다.

3. **통근 시간의 증가와 경제적 손실**:
   - 출퇴근 비율이 70%에 달하는 상황에서, 평균 속도가 낮고 대중교통 이용률이 낮으면 통근 시간이 크게 증가하게 됩니다. 이는 개인의 생활 질 저하뿐만 아니라, 경제적 손실로 이어질 수 있습니다. 장시간의 이동은 생산성 감소와 함께 지역 경제에 부정적인 영향을 미칠 수 있습니다.

이러한 문제들을 해결하기 위해서는 대중교통 시스템의 개선, 도로 인프라 확충, 그리고 교통수요 관리 전략이 필요합니다.


=== 2단계: 단계별 교통 대책 수립 ===
[단기 교통 대책 (1~2년 내 시행)]
1. **S-BRT 신설**: 신도시와 서울을 연결하는 고속 버스 전용 차로(S-BRT)를 신설하여 대중교통의 속도와 편리함을 개선합니다. 이를 통해 대중교통 이용률을 높이고, 출퇴근 시간의 혼잡도를 줄이는 효과를 기대할 수 있습니

3.도시계획 민원/법령 키워드 및 조항 추출기

In [3]:
from dotenv import load_dotenv
from langchain_core.prompts import (
    FewShotChatMessagePromptTemplate,
    ChatPromptTemplate
)
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

# 1. 예시 프롬프트 구성
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{complaint}"),
    ("ai", "카테고리: {category} | 관련법: {law} | 키워드: {keywords}")
])

# 2. 도시계획/교통 분야 예시 데이터셋
examples = [
    {
        "complaint": "아파트 단지 앞 교차로에 좌회전 신호 시간이 너무 짧아 출근길 차량 통행이 너무 심하게 막힙니다. 신호 주기 연장이 필요합니다.",
        "category": "교통",
        "law": "도로교통법",
        "keywords": "신호체계, 교차로 정체, 신호주기"
    },
    {
        "complaint": "우리 동네 주거지역 옆에 고층 지식산업센터가 들어서면서 일조권 침해와 조망권 피해가 심각합니다. 용도지역 변경 과정을 공개해주세요.",
        "category": "도시계획",
        "law": "국토의 계획 및 이용에 관한 법률",
        "keywords": "용도지역, 일조권, 주민공람"
    },
    {
        "complaint": "재개발 정비구역 지정 후 조합 설립 단계에서 추진위 승인 관련 절차상 하자가 있는 것 같습니다. 구청의 관리감독을 요청합니다.",
        "category": "정비사업",
        "law": "도시 및 주거환경정비법",
        "keywords": "재개발, 조합설립, 정비구역"
    }
]

# 3. Few-Shot 프롬프트 생성
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

# 4. 최종 통합 프롬프트
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 도시계획 및 교통 분야 민원 분류 전문가입니다. 입력된 민원글을 분석하여 반드시 지정된 형식('카테고리: ... | 관련법: ... | 키워드: 단어1, 단어2, 단어3')으로만 출력하세요."),
    few_shot_prompt,
    ("human", "{input}")
])

# 5. 체인 결합
chain = final_prompt | llm | StrOutputParser()

# 6. 테스트 민원 데이터
test_complaint = "신도시 S-BRT 전용차로 설치 공사 때문에 기존 일반차로가 줄어들어 상가 입주민들의 물품 하역차량 주차가 불가능해졌습니다. 하역공간 확보 및 ITS 기반 가변차로 도입을 검토해주십시오."

result = chain.invoke({"input": test_complaint})

print("--- 입력 민원 내용 ---")
print(test_complaint)
print("\n--- 자동 분류 및 키워드 추출 결과 ---")
print(result)

--- 입력 민원 내용 ---
신도시 S-BRT 전용차로 설치 공사 때문에 기존 일반차로가 줄어들어 상가 입주민들의 물품 하역차량 주차가 불가능해졌습니다. 하역공간 확보 및 ITS 기반 가변차로 도입을 검토해주십시오.

--- 자동 분류 및 키워드 추출 결과 ---
카테고리: 교통 | 관련법: 도로교통법 | 키워드: BRT, 하역공간, 가변차로
